In [ ]:
!pip install langgraph langchain-core
!pip install langchain-groq duckduckgo-search
!pip install langchain-community
!pip install -U ddgs

In [ ]:
!pip install tavily-python fredapi requests

In [ ]:
import requests
from tavily import TavilyClient
from fredapi import Fred
from google.colab import userdata

# Pulling secrets
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')
FRED_API_KEY = userdata.get('FRED_API_KEY')
ALPHA_VANTAGE_KEY = userdata.get('ALPHA_VANTAGE_KEY')

# Initialize Clients
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
fred_client = Fred(api_key=FRED_API_KEY)

print("API Clients Initialized Successfully.")

In [ ]:
from google.colab import userdata
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Initialize the Brain
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
print("Groq LLM Initialized Successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

LOAD_PATH = "/content/drive/MyDrive/colab_rag_data"

# Wipe stale local copies and restore fresh
for folder in ["./chroma_db", "./rag_data"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

shutil.copytree(f"{LOAD_PATH}/chroma_db", "./chroma_db")
shutil.copytree(f"{LOAD_PATH}/rag_data",  "./rag_data")

print("RAG data restored from Drive.")
print("rag_data contents:", os.listdir("./rag_data"))

# RAG Microservice (flask and ngrok)



In [ ]:
!pip install chromadb==1.5.8 tokenizers==0.15.2 flask pyngrok nest_asyncio

In [ ]:
!pip install llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-llms-langchain llama-index-retrievers-bm25 llama-index-postprocessor-flag-embedding-reranker llama-parse sentence-transformers

In [ ]:
!pip install git+https://github.com/FlagOpen/FlagEmbedding.git

In [ ]:
!pip install langgraph langchain-core langchain-groq langchain-community tavily-python fredapi requests duckduckgo-search

In [ ]:
!pip uninstall -y opentelemetry-sdk opentelemetry-api opentelemetry-semantic-conventions opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-http opentelemetry-proto opentelemetry-exporter-otlp-proto-common 2>/dev/null

!pip install opentelemetry-sdk==1.19.0 opentelemetry-api==1.19.0 opentelemetry-semantic-conventions==0.36b0 opentelemetry-exporter-otlp-proto-grpc==1.19.0 --force-reinstall

In [ ]:
!pip install "tokenizers==0.23.0" transformers -U --force-reinstall

In [ ]:
# 1. Fix the HuggingFace breaking change
!pip install "huggingface_hub==0.27.1" --force-reinstall

# 2. Fix the corrupted OpenTelemetry namespace
!pip install opentelemetry-api opentelemetry-sdk --force-reinstall

In [ ]:
!pip uninstall -y transformers tokenizers
!pip install transformers tokenizers --upgrade

In [ ]:
#RAG MICROSERVICE + NGROK
# Loads pre-built RAG data and serves it as a local API.
# Agent calls it over HTTP (like any external service)

import sys
import os
import time
import pickle
import threading
import psutil
from flask import Flask, request, jsonify
from pyngrok import ngrok
from google.colab import userdata
import nest_asyncio

#nest_asyncio to prevent event loop errors in Colab
nest_asyncio.apply()

# Kill old ngrok tunnels and free up port 5001 so we can rerun this cell safely
ngrok.kill()
for proc in psutil.process_iter(['pid', 'name']):
    try:
        for conn in proc.connections(kind='inet'):
            if conn.laddr.port == 5001:
                proc.kill()
    except:
        pass

import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Settings as LlamaSettings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.retrievers import VectorIndexRetriever, QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker
from llama_index.core.query_engine import RetrieverQueryEngine, TransformQueryEngine
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.langchain import LangChainLLM

# Verify saved data exists
assert os.path.exists("rag_data/nodes.pkl"), "nodes.pkl not found. Run the RAG notebook save cell first."
assert os.path.exists("rag_data/config.pkl"), "config.pkl not found. Run the RAG notebook save cell first."
assert os.path.exists("./chroma_db"), "chroma_db not found. Run the full RAG notebook first."

with open("rag_data/config.pkl", "rb") as f:
    rag_config = pickle.load(f)

with open("rag_data/nodes.pkl", "rb") as f:
    base_nodes = pickle.load(f)

print(f"Loaded {len(base_nodes)} nodes from disk.")
print(f"Using embed model: {rag_config['embed_model_name']}")

#RAGServer class
class RAGServer:
    """ Self-contained RAG pipeline.
    Holds query_engine in memory.
    Can rebuild itself when user uploads new docs.
    """
    def __init__(self, nodes, config, groq_llm):
        self.config = config
        self.nodes = nodes

        print("   Loading embedding model...")
        self.embed_model = HuggingFaceEmbedding(
            model_name=config["embed_model_name"],
            device="cpu"
        )

        LlamaSettings.embed_model = self.embed_model
        LlamaSettings.llm = LangChainLLM(llm=groq_llm)

        print("   Connecting to Chroma...")
        self._chroma_client = chromadb.PersistentClient(path="./chroma_db")

        self.query_engine = self._build_engine(nodes)
        print(f"   RAGServer ready. {len(nodes)} nodes indexed.")

    def _build_engine(self, nodes):
        """Builds hybrid retriever + reranker + HyDE pipeline from nodes."""
        collection = self._chroma_client.get_or_create_collection(self.config["collection_name"])
        vector_store = ChromaVectorStore(chroma_collection=collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        index = VectorStoreIndex.from_vector_store(
            vector_store=vector_store,
            storage_context=storage_context,
        )

        vec_retriever = VectorIndexRetriever(index=index, similarity_top_k=4)
        bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=4)
        hybrid_retriever = QueryFusionRetriever(
            [vec_retriever, bm25_retriever],
            similarity_top_k=4,
            num_queries=1,
            mode="reciprocal_rerank"
        )

        reranker = FlagEmbeddingReranker(model=self.config["reranker_model"], top_n=3)
        hyde = HyDEQueryTransform(include_original=True)

        base_engine = RetrieverQueryEngine.from_args(
            retriever=hybrid_retriever,
            node_postprocessors=[reranker],
            streaming=False
        )
        return TransformQueryEngine(query_engine=base_engine, query_transform=hyde)

    def query(self, text: str) -> dict:
        """Run a query. Returns answer + sources dict."""
        response = self.query_engine.query(text)
        answer = response.response if hasattr(response, "response") else str(response)

        sources = []
        for i, node in enumerate(response.source_nodes):
            meta = node.node.metadata or {}
            score = round(node.score, 3) if node.score else 0
            preview = node.node.get_text()[:200].replace("\n", " ")
            sources.append({
                "index": i + 1,
                "file": meta.get("file_name", meta.get("source", f"chunk_{i}")),
                "score": score,
                "preview": preview,
            })

        return {"answer": answer, "sources": sources}

    def _build_from_nodes(self, nodes):
        collection = self._chroma_client.get_or_create_collection(self.config["collection_name"])
        vector_store = ChromaVectorStore(chroma_collection=collection)
        storage_context = StorageContext.from_defaults(vector_store=vector_store)

        index = VectorStoreIndex(
            nodes,
            storage_context=storage_context,
            embed_model=self.embed_model
        )

        vec_retriever = VectorIndexRetriever(index=index, similarity_top_k=4)
        bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=4)
        hybrid_retriever = QueryFusionRetriever(
            [vec_retriever, bm25_retriever],
            similarity_top_k=4,
            num_queries=1,
            mode="reciprocal_rerank"
        )

        reranker = FlagEmbeddingReranker(model=self.config["reranker_model"], top_n=3)
        hyde = HyDEQueryTransform(include_original=True)

        base_engine = RetrieverQueryEngine.from_args(
            retriever=hybrid_retriever,
            node_postprocessors=[reranker],
            streaming=False
        )
        return TransformQueryEngine(query_engine=base_engine, query_transform=hyde)

    def rebuild_from_pdfs(self, pdf_dir: str) -> int:
        from llama_parse import LlamaParse
        from llama_index.core import SimpleDirectoryReader

        parser = LlamaParse(
            api_key=userdata.get("LLAMA_CLOUD_API_KEY"),
            result_type="markdown",
            verbose=False,
            num_workers=2
        )
        new_docs = SimpleDirectoryReader(pdf_dir, file_extractor={".pdf": parser}).load_data()

        node_parser = MarkdownNodeParser(
            chunk_size=self.config["chunk_size"],
            chunk_overlap=self.config["chunk_overlap"]
        )
        new_nodes = node_parser.get_nodes_from_documents(new_docs)

        self._chroma_client.delete_collection(self.config["collection_name"])

        self.nodes = new_nodes
        self.query_engine = self._build_from_nodes(new_nodes)

        with open("rag_data/nodes.pkl", "wb") as f:
            pickle.dump(new_nodes, f)

        return len(new_nodes)

# 3. Instantiate server (using the 'llm' variable from your previous cell)
print("\nBuilding RAG pipeline...")
rag_server = RAGServer(nodes=base_nodes, config=rag_config, groq_llm=llm)


# 4. Flask app
rag_app = Flask("rag_microservice")

@rag_app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "nodes_loaded": len(rag_server.nodes),
        "embed_model": rag_config["embed_model_name"],
    })

@rag_app.route("/query", methods=["POST"])
def handle_query():
    data = request.get_json()
    if not data or "query" not in data:
        return jsonify({"error": "No query provided"}), 400
    try:
        result = rag_server.query(data["query"])
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": f"{type(e).__name__}: {e}"}), 500

@rag_app.route("/upload", methods=["POST"])
def handle_upload():
    if "files" not in request.files:
        return jsonify({"error": "No files provided"}), 400

    os.makedirs("data_live", exist_ok=True)

    for old in os.listdir("data_live"):
        os.remove(f"data_live/{old}")

    saved = []
    for f in request.files.getlist("files"):
        path = f"data_live/{f.filename}"
        f.save(path)
        saved.append(f.filename)

    try:
        new_count = rag_server.rebuild_from_pdfs("data_live")
        return jsonify({"status": "rebuilt", "files": saved, "nodes_built": new_count})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Start Flask in background thread
def _run_flask():
    rag_app.run(host="127.0.0.1", port=5001, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# ngrok tunnel
NGROK_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5001).public_url
print(f"\nRAG server running:")
print(f"  Local:  http://127.0.0.1:5001")
print(f"  Public: {public_url}")
print(f"  Nodes:  {len(base_nodes)}")

RAG_BASE_URL = "http://127.0.0.1:5001"

# Quick health check
import requests as http_requests
h = http_requests.get(f"{RAG_BASE_URL}/health").json()
print("Health Check:", h)

In [ ]:
from langchain_core.tools import tool

@tool
def tavily_search_tool(query: str) -> str:
    """Use this tool for live web searches, recent news, and geopolitical events."""
    print(f"   [TOOL: TAVILY] Searching web for: '{query}'")
    try:
        # Advanced depth ensures it reads the actual websites, not just titles
        response = tavily_client.search(query=query, search_depth="advanced")
        # Extract the clean text summaries
        results = "\n".join([f"- {res['content']}" for res in response['results']])
        return f"Live Web Data for '{query}':\n{results}"
    except Exception as e:
        return f"Web search failed. Error: {e}"

@tool
def fred_macro_tool(series_id: str) -> str:
    """
    Fetches macroeconomic data from the Federal Reserve.
    Returns latest value AND derived metrics (YoY%, MoM%) for scoring.
    Requires a valid FRED Series ID.
    """
    print(f"   [TOOL: FRED] Fetching series: '{series_id}'")
    try:
        import pandas as pd
        from datetime import datetime, timedelta

        # Fetch 13 months to compute YoY
        end_date   = datetime.today()
        start_date = end_date - timedelta(days=500)

        data = fred_client.get_series(
            series_id,
            observation_start=start_date.strftime("%Y-%m-%d"),
            observation_end=end_date.strftime("%Y-%m-%d")
        )
        data = data.dropna()

        if len(data) == 0:
            return f"FRED Data for {series_id}: No data returned."

        latest_date  = data.index[-1].strftime("%Y-%m-%d")
        latest_value = round(float(data.iloc[-1]), 4)

        # Derived metrics
        # MoM % change (1 month ago)
        mom_pct = None
        if len(data) >= 2:
            prev_1m  = float(data.iloc[-2])
            mom_pct  = round(((latest_value - prev_1m) / abs(prev_1m)) * 100, 4) \
                       if prev_1m != 0 else None

        # 3M % change
        m3_pct = None
        if len(data) >= 4:
            prev_3m = float(data.iloc[-4])
            m3_pct  = round(((latest_value - prev_3m) / abs(prev_3m)) * 100, 4) \
                      if prev_3m != 0 else None

        # YoY % change (12 months ago)
        yoy_pct = None
        if len(data) >= 13:
            prev_12m = float(data.iloc[-13])
            yoy_pct  = round(((latest_value - prev_12m) / abs(prev_12m)) * 100, 4) \
                       if prev_12m != 0 else None

        # Format output — EXACT pattern _extract_fred_value parses
        # Primary line: always present
        output_lines = [
            f"FRED|{series_id}|LEVEL|{latest_value}|{latest_date}"
        ]
        if mom_pct  is not None:
            output_lines.append(f"FRED|{series_id}|MOM_PCT|{mom_pct}|{latest_date}")
        if m3_pct   is not None:
            output_lines.append(f"FRED|{series_id}|3M_PCT|{m3_pct}|{latest_date}")
        if yoy_pct  is not None:
            output_lines.append(f"FRED|{series_id}|YOY_PCT|{yoy_pct}|{latest_date}")

        # Human readable block underneath (for synthesizer LLM)
        human_lines = [
            f"FRED Data for {series_id} (As of {latest_date}):",
            f"  Level:      {latest_value}",
        ]
        if mom_pct  is not None: human_lines.append(f"  MoM %:      {mom_pct:+.2f}%")
        if m3_pct   is not None: human_lines.append(f"  3M %:       {m3_pct:+.2f}%")
        if yoy_pct  is not None: human_lines.append(f"  YoY %:      {yoy_pct:+.2f}%")

        return "\n".join(output_lines) + "\n\n" + "\n".join(human_lines)

    except Exception as e:
        return f"FRED API failed for {series_id}. Error: {type(e).__name__}: {e}"

@tool
def alpha_vantage_tool(symbol: str) -> str:
    """Fetches live market quotes for equities or ETFs (e.g., GLD for Gold)."""
    print(f"   [TOOL: ALPHA VANTAGE] Fetching quote for: '{symbol}'")
    try:
        url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={ALPHA_VANTAGE_KEY}"
        response = requests.get(url).json()
        quote = response.get("Global Quote", {})
        if not quote:
            return f"Alpha Vantage returned no data for {symbol}. Rate limit may be exceeded."
        price = quote.get("05. price", "Unknown")
        return f"Live Market Data for {symbol}: Current Price is ${price}"
    except Exception as e:
        return f"Alpha Vantage API failed. Error: {e}"

@tool
def rag_wgc_tool(query: str) -> str:
    """
    Search internal financial PDF documents — WGC reports, earnings,
    commodity research, any PDF the user has uploaded.
    Use for: gold demand data, ETF flows, central bank buying,
    mine supply, company earnings, analyst price targets.
    """
    print(f"   [TOOL: RAG] Querying internal docs: '{query}'")
    try:
        resp = http_requests.post(
            f"{RAG_BASE_URL}/query",
            json={"query": query},
            timeout=60
        )
        if resp.status_code != 200:
            return f"RAG server error {resp.status_code}: {resp.text}"

        data    = resp.json()
        answer  = data.get("answer", "No answer returned.")
        sources = data.get("sources", [])

        source_lines = [
            f"  [Doc {s['index']}] {s['file']} "
            f"(score: {s['score']})\n  {s['preview']}"
            for s in sources
        ]
        source_block = (
            "\n".join(source_lines)
            if source_lines
            else "  No matching documents found."
        )

        return (
            f"Internal Document Analysis for '{query}':\n"
            f"{answer}\n\n"
            f"Source Evidence:\n{source_block}"
        )

    except http_requests.exceptions.ConnectionError:
        return (
            "RAG server not running. "
            "Run CELL 4.5 first to start the RAG microservice."
        )
    except Exception as e:
        return f"RAG tool error: {type(e).__name__}: {e}"

# Mapping the tools
tool_map = {
    "tavily_search": tavily_search_tool,
    "fred_macro": fred_macro_tool,
    "alpha_vantage": alpha_vantage_tool,
    "rag_wgc": rag_wgc_tool
}
print("Robust Pro Tools successfully loaded into memory!")

**#define state**

In [ ]:
from typing import TypedDict, Annotated, List
import operator
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

#our agent's clipboard. It holds all the data as it moves through the steps.
# 1. STATE
class ResearchState(TypedDict):
    question: str
    plan: list[dict]
    tool_outputs: Annotated[list, operator.add]
    correlation_matrix: dict
    final_report: str

# 2. SCHEMAS
class ToolCall(BaseModel):
    tool_name: str = Field(description="Must be one of: tavily_search, fred_macro, alpha_vantage, rag_wgc")
    tool_query: str = Field(description="Search query, ticker (e.g. GC=F, DX-Y.NYB), or FRED Series ID (e.g. DFII10, CPIAUCSL).")

class RoutePlan(BaseModel):
    plan: List[ToolCall] = Field(description="The list of tools to use and their queries.")


In [ ]:
#7.5 CORRELATION ENGINE (less llm and more logic)

import re
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

#1. FACTOR REGISTRY
# metric_to_use tells the parser WHICH derived value to pull:
#   LEVEL   = raw series value  (real yields, fed funds, VIX)
#   YOY_PCT = year-over-year %  (CPI — inflation rate)
#   3M_PCT  = 3-month % change  (dollar index — momentum matters)
#   MOM_PCT = month-over-month  (housing starts — level + momentum)

@dataclass
class GoldFactor:
    name:              str
    fred_series:       str
    metric_to_use:     str    # LEVEL | YOY_PCT | 3M_PCT | MOM_PCT
    direction:         int    # +1 = high value bullish gold, -1 = bearish
    weight:            int    # 1=supporting, 2=significant, 3=dominant
    threshold_neutral: float  # center of the scoring range
    threshold_extreme: float  # value where signal is maxed out
    historical_mean:   float  # for z-score normalization
    historical_std:    float
    lag_weeks:         int
    description:       str

GOLD_FACTOR_REGISTRY: dict[str, GoldFactor] = {

    "real_yields": GoldFactor(
        name             = "10Y Real Yield (TIPS)",
        fred_series      = "DFII10",
        metric_to_use    = "LEVEL",       # raw % level is correct here
        direction        = -1,            # higher real yield = bearish gold
        weight           = 3,
        threshold_neutral= 0.5,           # ~Fed neutral real rate
        threshold_extreme= 2.5,           # meaningfully restrictive
        historical_mean  = 0.48,          # TIPS avg 2003-2024
        historical_std   = 1.21,
        lag_weeks        = 1,
        description      = "Opportunity cost of holding gold. Dominant driver."
    ),

    "dollar": GoldFactor(
        name             = "Broad Dollar Index (Trade-Weighted)",
        fred_series      = "DTWEXBGS",
        metric_to_use    = "3M_PCT",      # % change — momentum is what matters
        direction        = -1,            # dollar strengthening = bearish gold
        weight           = 3,
        threshold_neutral= 0.0,           # flat dollar = neutral
        threshold_extreme= 3.0,           # 3% 3M move = strong signal
        historical_mean  = 0.12,          # avg 3M % change historically
        historical_std   = 2.1,
        lag_weeks        = 0,
        description      = "USD strength inversely correlated with gold price."
    ),

    "cpi": GoldFactor(
        name             = "CPI Inflation (YoY %)",
        fred_series      = "CPIAUCSL",
        metric_to_use    = "YOY_PCT",     # derive YoY — NOT the index level
        direction        = +1,            # higher inflation = bullish gold
        weight           = 2,
        threshold_neutral= 2.5,           # Fed target
        threshold_extreme= 5.0,           # meaningfully high inflation
        historical_mean  = 2.8,           # avg YoY CPI 1990-2024
        historical_std   = 1.9,
        lag_weeks        = 6,
        description      = "Gold as inflation hedge. Acts with lag."
    ),

    "vix": GoldFactor(
        name             = "VIX Fear Index",
        fred_series      = "VIXCLS",
        metric_to_use    = "LEVEL",       # raw level is correct
        direction        = +1,            # fear spike = safe haven demand
        weight           = 2,
        threshold_neutral= 20.0,
        threshold_extreme= 35.0,
        historical_mean  = 19.5,
        historical_std   = 8.2,
        lag_weeks        = 0,
        description      = "Safe haven demand. Immediate effect."
    ),

    "housing": GoldFactor(
        name             = "Housing Starts",
        fred_series      = "HOUST",
        metric_to_use    = "MOM_PCT",     # momentum more signal than level
        direction        = -1,            # construction boom = risk-on, less gold
        weight           = 1,
        threshold_neutral= 0.0,
        threshold_extreme= 5.0,           # 5% MoM = strong momentum
        historical_mean  = 0.3,
        historical_std   = 7.5,           # housing starts are volatile MoM
        lag_weeks        = 8,
        description      = "Cross-sector capital flow proxy. Weak signal."
    ),

    "fedfunds": GoldFactor(
        name             = "Federal Funds Rate",
        fred_series      = "FEDFUNDS",
        metric_to_use    = "LEVEL",       # raw rate level is correct
        direction        = -1,            # higher rates = bearish gold
        weight           = 2,
        threshold_neutral= 2.5,
        threshold_extreme= 5.5,
        historical_mean  = 2.1,
        historical_std   = 2.3,
        lag_weeks        = 2,
        description      = "Nominal rate environment. Correlated with real yields."
    ),
}
# 2 PARSER — reads pipe-delimited FRED tool output

def _extract_fred_metric(
    tool_outputs: list[str],
    series_id:    str,
    metric:       str          # LEVEL | YOY_PCT | 3M_PCT | MOM_PCT
) -> Optional[float]:
    """
    Parses the structured pipe-delimited lines from fred_macro_tool.
    Format: FRED|SERIES_ID|METRIC|VALUE|DATE
    Falls back to human-readable line if structured line missing.
    """
    target_pattern = f"FRED|{series_id}|{metric}|"

    for output in tool_outputs:
        # Primary: parse structured pipe line
        for line in output.splitlines():
            if line.startswith(target_pattern):
                parts = line.split("|")
                if len(parts) >= 4:
                    try:
                        val = float(parts[3])
                        # Sanity gate: reject year-like values
                        if 1900 <= val <= 2100:
                            continue
                        return val
                    except ValueError:
                        continue
        # Fallback: parse old human-readable format if pipe lines absent
        # "FRED Data for SERIES_ID (As of DATE): VALUE"
        if metric == "LEVEL" and series_id in output:
            match = re.search(
                rf"FRED Data for {re.escape(series_id)}"
                rf"\s*\(As of [^)]+\):\s*([-+]?\d+\.?\d*)",
                output
            )
            if match:
                try:
                    val = float(match.group(1))
                    if not (1900 <= val <= 2100):
                        return val
                except ValueError:
                    pass
    return None

#3 SIGNAL SCORER

@dataclass
class FactorSignal:
    factor_key:       str
    series_id:        str
    metric_used:      str
    raw_value:        float
    z_score:          float
    continuous_score: float
    display_score:    int       # rounded for display
    direction_label:  str       # BULLISH | BEARISH | NEUTRAL
    confidence:       str       # HIGH | MEDIUM | LOW
    note:             str

def score_factor(factor_key: str, raw_value: float) -> FactorSignal:
    """
    Scores a single factor using z-score normalization + tanh bounding.
    Z-score: how unusual is this value vs history?
    tanh:    smooth, bounded output — no cliff edges between buckets.
    """
    f = GOLD_FACTOR_REGISTRY[factor_key]

    # Z-score normalization
    z = (raw_value - f.historical_mean) / f.historical_std \
        if f.historical_std != 0 else 0.0
    z = round(z, 3)

    # Continuous gold impact score
    # tanh(z * 0.6): z=±1 → ±0.54, z=±2 → ±0.83, z=±3 → ±0.96
    # Multiplied by direction and weight gives range (-weight, +weight)
    continuous = float(np.tanh(z * 0.6)) * f.direction * f.weight
    continuous = round(continuous, 3)

    # Confidence from z magnitude
    if abs(z) >= 1.5:
        confidence = "HIGH"
    elif abs(z) >= 0.6:
        confidence = "MEDIUM"
    else:
        confidence = "LOW"

    display_score = int(round(continuous))
    display_score = max(-3, min(3, display_score))

    if display_score > 0:
        direction_label = "BULLISH"
    elif display_score < 0:
        direction_label = "BEARISH"
    else:
        direction_label = "NEUTRAL"

    note = (
        f"{f.metric_to_use} value: {raw_value} | "
        f"z={z:+.2f} | "
        f"hist mean={f.historical_mean}, std={f.historical_std} | "
        f"lag ~{f.lag_weeks}w"
    )

    return FactorSignal(
        factor_key       = factor_key,
        series_id        = f.fred_series,
        metric_used      = f.metric_to_use,
        raw_value        = raw_value,
        z_score          = z,
        continuous_score = continuous,
        display_score    = display_score,
        direction_label  = direction_label,
        confidence       = confidence,
        note             = note,
    )

#4 CORRELATION MATRIX NODE

@dataclass
class CorrelationMatrix:
    signals:           list       = field(default_factory=list)
    net_score:         float      = 0.0
    data_confidence:   float      = 0.0
    verdict:           str        = "NEUTRAL"
    regime:            str        = "AMBIGUOUS"
    trigger_condition: str        = ""
    scorecard_text:    str        = ""


def build_correlation_matrix(state: dict) -> dict:
    print("-> [CORRELATION] Scoring macro factors...")

    tool_outputs = state.get("tool_outputs", [])
    signals      = []

    for key, factor in GOLD_FACTOR_REGISTRY.items():
        value = _extract_fred_metric(
            tool_outputs,
            factor.fred_series,
            factor.metric_to_use
        )

        if value is None:
            print(f"   [CORRELATION] No {factor.metric_to_use} data "
                  f"for {factor.fred_series} — skipping")
            continue

        signal = score_factor(key, value)
        signals.append(signal)
        print(
            f"   {factor.fred_series} ({factor.metric_to_use}): "
            f"{value} | z={signal.z_score:+.2f} | "
            f"{signal.direction_label} [{signal.display_score:+d}]"
        )

    if not signals:
        matrix = CorrelationMatrix(
            verdict           = "NEUTRAL",
            trigger_condition = "No FRED data parsed. Check tool outputs.",
            scorecard_text    = "Insufficient data.",
            data_confidence   = 0.0,
        )
        return {"correlation_matrix": matrix}

    # Continuous net score
    net_score        = round(sum(s.continuous_score for s in signals), 3)
    data_confidence  = round(len(signals) / len(GOLD_FACTOR_REGISTRY), 2)

    # Verdict
    if   net_score >=  4.0: verdict = "STRONG BULLISH"
    elif net_score >=  1.5: verdict = "BULLISH"
    elif net_score <= -4.0: verdict = "STRONG BEARISH"
    elif net_score <= -1.5: verdict = "BEARISH"
    else:                   verdict = "NEUTRAL"

    # Regime
    regime    = _detect_regime(signals)
    trigger   = _build_trigger(signals, net_score)
    scorecard = _build_scorecard(signals, net_score, data_confidence)

    matrix = CorrelationMatrix(
        signals           = signals,
        net_score         = net_score,
        data_confidence   = data_confidence,
        verdict           = verdict,
        regime            = regime,
        trigger_condition = trigger,
        scorecard_text    = scorecard,
    )
    return {"correlation_matrix": matrix}


# 5 REGIME DETECTOR

def _detect_regime(signals: list) -> str:
    sc = {s.factor_key: s.continuous_score for s in signals}

    ry  = sc.get("real_yields", 0)
    dxy = sc.get("dollar",      0)
    cpi = sc.get("cpi",         0)
    vix = sc.get("vix",         0)

    if cpi > 1.0 and ry < -1.0:
        return "STAGFLATION — high inflation + falling real yields: strong gold tailwind"
    if cpi > 1.0 and ry > 0.5:
        return "REFLATION — inflation rising but yields keeping pace: mixed for gold"
    if ry > 0.8 and dxy > 0.8 and vix > 0.5:
        return "RISK-OFF — yields falling, dollar weak, fear elevated: gold positive"
    if ry < -0.8 and dxy < -0.8:
        return "RISK-ON — yields rising, dollar strong: gold headwind"
    return "AMBIGUOUS — no dominant macro regime"


#6 TRIGGER CONDITION

def _build_trigger(signals: list, net_score: float) -> str:
    if not signals:
        return "No data."

    bearish = [s for s in signals if s.continuous_score < 0]
    bullish = [s for s in signals if s.continuous_score > 0]
    f_reg   = GOLD_FACTOR_REGISTRY

    if net_score < 0 and bearish:
        worst = min(bearish, key=lambda x: x.continuous_score)
        f     = f_reg[worst.factor_key]
        return (
            f"WATCH [{f.fred_series} {worst.metric_used}]: "
            f"Currently {worst.raw_value} (z={worst.z_score:+.2f}, bearish). "
            f"A reversal toward historical mean ({f.historical_mean}) "
            f"would add ~{abs(worst.continuous_score):.1f} to net score. "
            f"Expected lag: ~{f.lag_weeks} weeks."
        )
    elif net_score > 0 and bullish:
        best = max(bullish, key=lambda x: x.continuous_score)
        f    = f_reg[best.factor_key]
        return (
            f"WATCH [{f.fred_series} {best.metric_used}]: "
            f"Currently {best.raw_value} (z={best.z_score:+.2f}, bullish). "
            f"A reversal toward historical mean ({f.historical_mean}) "
            f"would subtract ~{abs(best.continuous_score):.1f} from net score. "
            f"Expected lag: ~{f.lag_weeks} weeks."
        )
    else:
        dominant = max(signals, key=lambda x: abs(x.continuous_score))
        f        = f_reg[dominant.factor_key]
        return (
            f"NEUTRAL ZONE — balanced signals. "
            f"Swing factor: {f.fred_series} ({dominant.metric_used}={dominant.raw_value}). "
            f"A 1-std move ({f.historical_std}) would break the tie."
        )


#7 SCORECARD FORMATTER

def _build_scorecard(
    signals:         list,
    net_score:       float,
    data_confidence: float
) -> str:
    lines = []
    sorted_signals = sorted(
        signals,
        key=lambda x: abs(x.continuous_score),
        reverse=True
    )

    for s in sorted_signals:
        bar    = "█" * abs(s.display_score) + "░" * (3 - abs(s.display_score))
        prefix = "+" if s.continuous_score > 0 else ""
        f      = GOLD_FACTOR_REGISTRY[s.factor_key]
        lines.append(
            f"  {f.fred_series:<12} {s.metric_used:<8} "
            f"{str(s.raw_value):<10} "
            f"z={s.z_score:+.2f}  "
            f"→ {s.direction_label:<10} "
            f"[{prefix}{s.display_score}]  {bar}"
        )

    lines.append(
        f"\n  NET SCORE: {net_score:+.3f}  |  "
        f"DATA CONFIDENCE: {data_confidence:.0%}  |  "
        f"SIGNALS: {len(signals)}/{len(GOLD_FACTOR_REGISTRY)}"
    )
    return "\n".join(lines)

print("Correlation engine loaded.")
print("Factors registered:", list(GOLD_FACTOR_REGISTRY.keys()))

*Build nodes

In [ ]:
# 3 ROUTER NODE
def router_node(state: ResearchState):
    print(f"-> [ROUTER] Categorizing and planning for: '{state['question']}'")

    system_prompt = """You are an elite quantitative AI routing agent.
    Analyze the question.

    IF THE QUERY IS ABOUT GOLD/COMMODITIES, YOU MUST INCLUDE ALL OF THESE EXACTLY:
    1. alpha_vantage: 'GLD' (Gold ETF price)
    2. alpha_vantage: 'DX-Y.NYB' (US Dollar Index)
    3. fred_macro: 'DFII10' (10-Year Real Yields / TIPS)
    4. fred_macro: 'CPIAUCSL' (CPI Inflation)
    5. fred_macro: 'HOUST' (Housing Starts)
    6. fred_macro: 'FEDFUNDS' (Federal Funds Rate)
    7. fred_macro: 'VIXCLS' (VIX Fear Index)
    8. fred_macro: 'DTWEXBGS' (Broad Dollar Index trade-weighted)
    9. tavily_search: General news query about the commodity
    10. rag_wgc: Internal institutional documents query

    Items 1 through 8 are MANDATORY for every gold/commodity query.
    Do not skip any of them regardless of what the question focuses on.

    IF THE QUERY IS AN EQUITY (STOCK):
    Use alpha_vantage for the ticker, tavily_search for company news, and rag_wgc for earnings reports.
    """
    prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("user", "{question}")])

    structured_llm = llm.with_structured_output(RoutePlan)
    result = (prompt | structured_llm).invoke({"question": state["question"]})
    final_plan = [{"tool_name": tc.tool_name, "tool_query": tc.tool_query} for tc in result.plan]
    return {"plan": final_plan}

# 2. The Tool Execution Node
def execute_tools_node(state: ResearchState):
    print("-> [TOOLS] Executing the optimized plan...")
    outputs = []

    # We now loop through dictionaries, not just strings
    for task in state["plan"]:
        tool_name = task["tool_name"]
        tool_query = task["tool_query"]

        if tool_name in tool_map:
            tool_function = tool_map[tool_name]
            print(f"   [TOOLS] Running {tool_name} with specific query: '{tool_query}'")

            # INSTEAD of state["question"], we pass the optimized query
            result = tool_function.invoke(tool_query)

            formatted_result = f"--- Data from {tool_name} (Query: {tool_query}) ---\n{result}\n"
            outputs.append(formatted_result)
        else:
            print(f"   [WARNING] Tool '{tool_name}' not found.")

    return {"tool_outputs": outputs}


In [ ]:
# We define the strict layout of our final Investment Memo
class ForecastMemo(BaseModel):
    asset_analyzed: str = Field(description="The primary equity or commodity analyzed.")
    raw_technicals: str = Field(description="Exact numbers only: Current price, past price, and % change.")
    raw_macro_and_news: str = Field(description="Bullet points of the exact FRED rates/numbers or specific news headlines found. Do not summarize too heavily; show the evidence.")
    institutional_context: str = Field(description="Summary of evidence from RAG. Say 'No internal documents provided' if empty.")
    ai_rationale: str = Field(description="Your synthesized conclusion. Tie the raw evidence together to explain what is happening.")
    directional_forecast: str = Field(description="MUST be one of: [BULLISH, BEARISH, NEUTRAL].")

#3. Synthesizer Node
def synthesizer_node(state: ResearchState):
    print("-> [SYNTHESIZER] Generating conclusion from scored signals...")

    raw_telemetry  = "\n".join(state["tool_outputs"])
    matrix: CorrelationMatrix = state["correlation_matrix"]

    # LLM only writes the rationale — verdict is already decided
    system_prompt = """You are a ruthless quant analyst.
    The signal scorecard and net verdict have already been computed deterministically.
    Your ONLY job: write 3–5 sentences of hard-hitting rationale explaining WHY
    the macro factors are producing this verdict. Cite specific numbers.
    Do not repeat the scorecard. Do not hedge. Be direct.

    CRITICAL: Use ONLY the FRED pipe-delimited data and Alpha Vantage
    prices as ground truth. Tavily articles are context only —
    never cite their specific numbers as current prices."""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", (
            "Question: {question}\n"
            "Net verdict: {verdict} (score: {net_score})\n"
            "Scored signals:\n{scorecard}\n"
            "Raw data:\n{raw_telemetry}"
        ))
    ])

    # Simpler schema now — LLM only owns rationale
    class RationaleOnly(BaseModel):
        ai_rationale: str = Field(description="3-5 sentence synthesis. Numbers only. No fluff.")

    structured_llm = llm.with_structured_output(RationaleOnly)
    memo = (prompt | structured_llm).invoke({
        "question":      state["question"],
        "verdict":       matrix.verdict,
        "net_score":     matrix.net_score,
        "scorecard":     matrix.scorecard_text,
        "raw_telemetry": raw_telemetry,
    })

    final_output = f"""
══════════════════════════════════════════════════════
CIE ANALYTICAL ENGINE  |  DATA CONFIDENCE: {matrix.data_confidence:.0%}
══════════════════════════════════════════════════════
► PART 1: RAW API TELEMETRY
{raw_telemetry}

──────────────────────────────────────────────────────
► PART 2: SIGNAL SCORECARD  (deterministic, no LLM)
{matrix.scorecard_text}

──────────────────────────────────────────────────────
► PART 3: AI RATIONALE  (synthesis only)
{memo.ai_rationale}

──────────────────────────────────────────────────────
► FINAL VERDICT:  {matrix.verdict}

► TRIGGER CONDITION:
  {matrix.trigger_condition}
══════════════════════════════════════════════════════
"""
    return {"final_report": final_output}


# Wire the Graph Together

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. Initialize the graph with our exact State schema
workflow = StateGraph(ResearchState)

# 2. Add the worker nodes we created
workflow.add_node("router", router_node)
workflow.add_node("execute_tools", execute_tools_node)
workflow.add_node("correlation", build_correlation_matrix)
workflow.add_node("synthesizer", synthesizer_node)

# 3. Define the exact path (Linear for now: Router -> Tools -> Synthesizer)
workflow.add_edge(START, "router")
workflow.add_edge("router", "execute_tools")
workflow.add_edge("execute_tools", "correlation")
workflow.add_edge("correlation",   "synthesizer")
#workflow.add_edge("execute_tools", "synthesizer")
workflow.add_edge("synthesizer", END)

# 4. Compile the graph into a runnable application
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print("\n CIE GRAPH COMPILED: QUANT MATRIX ACTIVE. ")

# *skeleton:

In [ ]:
config = {"configurable": {"thread_id": "test_thread_01"}}
q = {"question": "What is the current federal funds rate, and how is the current rate environment impacting housing construction starts and the price of the Gold ETF (GLD)?"}
state = app.invoke(q, config=config)
print(state['final_report'])

**Final Query cell.**



In [ ]:
#FINAL CELL (Production Query Interface)
import uuid

def run_cie_query(question: str, thread_id: str = None) -> dict:
    """
    Single entry point for all queries.
    Args:
        question:  Any financial question — gold, equities, macro
        thread_id: Optional. Pass same ID to continue a conversation.
                   Leave None to auto-generate a fresh session.
    Returns:
        dict with keys: final_report, verdict, net_score,
                        regime, trigger, confidence, thread_id
    """
    # Auto-generate thread if not provided
    if thread_id is None:
        thread_id = f"cie_{uuid.uuid4().hex[:8]}"

    # Correct initial state — all keys present
    initial_state = {
        "question":           question,
        "plan":               [],
        "tool_outputs":       [],
        "correlation_matrix": {},   # populated by correlation node
        "final_report":       ""
    }

    config = {"configurable": {"thread_id": thread_id}}

    print(f"\n{'═'*55}")
    print(f"CIE ENGINE QUERY")
    print(f"Thread:   {thread_id}")
    print(f"Question: {question}")
    print(f"{'═'*55}\n")

    try:
        final_state = app.invoke(initial_state, config=config)

        # ── Extract structured data for Streamlit later ─────
        matrix = final_state.get("correlation_matrix", {})

        # Handle both dataclass and dict (depending on how node returns it)
        if hasattr(matrix, "verdict"):
            verdict   = matrix.verdict
            net_score = matrix.net_score
            regime    = matrix.regime
            trigger   = matrix.trigger_condition
            scorecard = matrix.scorecard_text
            confidence= matrix.data_confidence
        else:
            verdict   = matrix.get("verdict",           "N/A")
            net_score = matrix.get("net_score",          0)
            regime    = matrix.get("regime",             "N/A")
            trigger   = matrix.get("trigger_condition", "N/A")
            scorecard = matrix.get("scorecard_text",    "N/A")
            confidence= matrix.get("data_confidence",    0)

        result = {
            "thread_id":    thread_id,
            "question":     question,
            "final_report": final_state.get("final_report", ""),
            "verdict":      verdict,
            "net_score":    net_score,
            "regime":       regime,
            "trigger":      trigger,
            "scorecard":    scorecard,
            "confidence":   confidence,
            "tool_outputs": final_state.get("tool_outputs", []),
        }

        print(final_state.get("final_report", "No report generated."))
        return result

    except Exception as e:
        print(f"Query failed: {type(e).__name__}: {e}")
        raise

# Demo queries — run any one of these
# Gold macro analysis
result_gold = run_cie_query(
    question="What is the current federal funds rate, and how is "
             "the current rate environment impacting housing "
             "construction starts and the price of Gold (GLD)?"
)

In [ ]:
# Uncomment to test equity:
# result_tsla = run_cie_query(
#     question="Analyze Tesla (TSLA). What is the 1-month price "
#              "trend and recent news headwinds or tailwinds?"
# )

# Uncomment to continue a conversation on same thread:
# result_followup = run_cie_query(
#     question="Given that analysis, what would a rate cut mean for gold?",
#     thread_id=result_gold["thread_id"]   # same thread = memory intact
# )

In [ ]:
# Cell A – Gold analysis
result = run_cie_query("What's the outlook for gold given current real yields?")

# Cell B – Follow-up (same thread, memory preserved)
follow = run_cie_query("How would a rate cut change that?", thread_id=result["thread_id"])

# Cell C – Tesla analysis (new thread)
run_cie_query("Analyze TSLA recent price action and news.")

In [ ]:
#for a chat loop
#("exit" to stop)
thread = None
while True:
    q = input("\nYour question (or 'exit'): ")
    if q.lower() == 'exit':
        break
    result = run_cie_query(q, thread_id=thread)
    thread = result["thread_id"]

UI


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
import uuid

#  UI Elements
question_input = widgets.Textarea(
    placeholder='e.g. How are real yields impacting gold prices right now?',
    layout=widgets.Layout(width='100%', height='80px')
)
run_button = widgets.Button(
    description='▶ Run Analysis',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)
output_area = widgets.Output()

# Callback
def on_button_click(b):
    with output_area:
        clear_output(wait=True)
        question = question_input.value.strip()
        if not question:
            print("⚠️ Please enter a question first.")
            return

        print(f"Analyzing: {question}\n")
        try:

            result = run_cie_query(question)
            verdict    = result['verdict']
            net_score  = result['net_score']
            confidence = result['confidence']
            scorecard  = result.get('scorecard', '')
            regime     = result.get('regime', '')
            trigger    = result.get('trigger', '')
            final_report = result.get('final_report', '')

            # Color-coded verdict
            color = 'green' if 'BULL' in verdict else 'red' if 'BEAR' in verdict else 'gray'
            display(Markdown(f"## Verdict: <span style='color:{color}'>{verdict}</span>"))
            display(Markdown(f"**Net Score:** {net_score:+.2f} | **Data Confidence:** {int(confidence*100)}%"))

            # Scorecard
            display(Markdown("### 📊 Signal Scorecard"))
            print(scorecard)

            display(Markdown(f"**Regime:** {regime}"))
            display(Markdown(f"**Trigger:** {trigger}"))

            # Extract rationale from final report
            if "► AI RATIONALE" in final_report:
                rationale = final_report.split("► AI RATIONALE")[1].split("──────")[0].strip()
                display(Markdown("### 🧠 AI Rationale"))
                print(rationale)
            else:
                display(Markdown("### Full Report"))
                print(final_report)

        except Exception as e:
            print(f"❌ Error: {e}")

run_button.on_click(on_button_click)

#Display UI
display(widgets.VBox([
    widgets.HTML("<h3>📊 CIE — Commodity Intelligence Engine</h3>"),
    question_input,
    run_button,
    output_area
]))